# Pima Indians Diabetes Classification Mini Project

This notebook constructs classification models on the Pima Indians Diabetes dataset from the UCI Repository. The objective is to predict whether a patient is diabetic or not diabetic using two supervised machine learning algorithms:

- K-Nearest Neighbors (KNN)
- Decision Tree

The models are evaluated using accuracy and confusion matrix, as required in the corrected problem statement.


## 1. Import Required Libraries

This cell imports the libraries required for data handling, visualization, preprocessing, model training, and evaluation. Pandas and NumPy are used for working with the dataset, Matplotlib and Seaborn are used for plotting, and Scikit-learn is used for machine learning models and metrics.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix


## 2. Load the Dataset

This cell loads the local `diabetes.csv` file. The dataset contains medical measurements such as glucose level, blood pressure, BMI, insulin level, age, and the final `Outcome` column.


In [ ]:
df = pd.read_csv('diabetes.csv')
df.head()


## 3. Display Dataset Size and Preview

This cell displays the shape of the dataset and previews the first five records. This output can be used as the dataset screenshot in the implementation chapter of the report.


In [ ]:
print('Dataset shape:', df.shape)
df.head()


## 4. Understand Dataset Information

This cell shows column data types, non-null counts, and descriptive statistics. It helps understand the range and distribution of the medical attributes before preprocessing.


In [ ]:
df.info()
df.describe()


## 5. Check Class Distribution

This cell checks how many records belong to each class in the `Outcome` column. `0` represents not diabetic and `1` represents diabetic.


In [ ]:
print(df['Outcome'].value_counts())

sns.countplot(x='Outcome', data=df, palette='Set2')
plt.title('Distribution of Diabetes Outcome')
plt.xlabel('Outcome (0 = Not Diabetic, 1 = Diabetic)')
plt.ylabel('Count')
plt.show()


## 6. Data Cleaning

Some medical columns contain zero values that are not realistic, such as zero glucose, blood pressure, insulin, or BMI. This cell replaces such invalid zero values with missing values and then fills them using the median of each column.


In [ ]:
columns_with_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in columns_with_zero:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

print(df[columns_with_zero].describe())


## 7. Split Features and Target

This cell separates the independent variables from the target variable. `X` contains the input features used for prediction, and `y` contains the `Outcome` class label.


In [ ]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

print('Feature columns:', list(X.columns))
print('Target column: Outcome')


## 8. Train-Test Split

This cell splits the dataset into training and testing sets. The training set is used to train the models, while the testing set is used to evaluate model performance.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])


## 9. Feature Scaling for KNN

KNN is distance-based, so features with larger numerical ranges can dominate the prediction. This cell standardizes the feature values so that each feature contributes fairly to the KNN model.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 10. Train the KNN Model

This cell creates and trains the K-Nearest Neighbors classifier. The model predicts the class of a test record based on the majority class among its nearest neighbors.


In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=11)
knn_model.fit(X_train_scaled, y_train)

knn_pred = knn_model.predict(X_test_scaled)


## 11. Evaluate KNN Using Accuracy and Confusion Matrix

This cell calculates the KNN accuracy and displays the KNN confusion matrix. Accuracy shows the percentage of correct predictions, while the confusion matrix shows correct and incorrect classifications for both classes.


In [ ]:
knn_accuracy = accuracy_score(y_test, knn_pred)
knn_cm = confusion_matrix(y_test, knn_pred)

print('KNN Accuracy:', knn_accuracy)

sns.heatmap(knn_cm, annot=True, fmt='d', cmap='Blues')
plt.title('KNN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


## 12. Train the Decision Tree Model

This cell creates and trains the Decision Tree classifier. A Decision Tree makes predictions by splitting the data into branches based on feature values.


In [ ]:
dt_model = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=4,
    min_samples_split=8,
    min_samples_leaf=4,
    random_state=42
)
dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)


## 13. Evaluate Decision Tree Using Accuracy and Confusion Matrix

This cell calculates the Decision Tree accuracy and displays its confusion matrix. These outputs are used to compare the Decision Tree model with the KNN model.


In [ ]:
dt_accuracy = accuracy_score(y_test, dt_pred)
dt_cm = confusion_matrix(y_test, dt_pred)

print('Decision Tree Accuracy:', dt_accuracy)

sns.heatmap(dt_cm, annot=True, fmt='d', cmap='Greens')
plt.title('Decision Tree Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


## 14. Compare Model Accuracy

This cell creates a comparison table and bar chart for the accuracy of KNN and Decision Tree. This helps identify which model performs better on the test data.


In [ ]:
results = pd.DataFrame({
    'Model': ['KNN', 'Decision Tree'],
    'Accuracy': [knn_accuracy, dt_accuracy]
})

print(results)

sns.barplot(x='Model', y='Accuracy', data=results, palette='Set2')
plt.title('Accuracy Comparison: KNN vs Decision Tree')
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.show()


## 15. Prediction on New Patient Data

This optional cell demonstrates how the trained KNN and Decision Tree models can predict diabetes for a new patient record. The KNN model uses scaled input, while the Decision Tree model uses the original feature values.


In [ ]:
sample_patient = pd.DataFrame([{
    'Pregnancies': 2,
    'Glucose': 120,
    'BloodPressure': 70,
    'SkinThickness': 25,
    'Insulin': 80,
    'BMI': 28.5,
    'DiabetesPedigreeFunction': 0.45,
    'Age': 35
}])

sample_patient_scaled = scaler.transform(sample_patient)

knn_sample_pred = knn_model.predict(sample_patient_scaled)[0]
dt_sample_pred = dt_model.predict(sample_patient)[0]

print('KNN Prediction:', 'Diabetic' if knn_sample_pred == 1 else 'Not Diabetic')
print('Decision Tree Prediction:', 'Diabetic' if dt_sample_pred == 1 else 'Not Diabetic')


## 16. Conclusion Notes

This notebook implemented KNN and Decision Tree classification models on the Pima Indians Diabetes dataset. Both models were evaluated using accuracy and confusion matrix. The final comparison table and graph show which model achieved better accuracy on the test set.
